In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json

w = WorkspaceClient()

app_name = "databrck-weather-vector-search"

# 1. Get the OAuth client ID for this Databricks App
app_client_id = w.apps.get(app_name).oauth2_app_client_id

# 2. Get the notebook's current internal Databricks token
notebook_token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

# 3. Exchange it for a token scoped specifically to this app
token_response = requests.post(
    f"{w.config.host}/oidc/v1/token",
    data={
        "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
        "subject_token": notebook_token,
        "subject_token_type":
            "urn:databricks:params:oauth:token-type:personal-access-token",
        "requested_token_type":
            "urn:ietf:params:oauth:token-type:access_token",
        "scope": "all-apis",
        "audience": app_client_id,
    },
)

token_response.raise_for_status()
app_token = token_response.json()["access_token"]

# 4. Call your Flask API
# Get the app URL dynamically instead of hardcoding
app_info = w.apps.get(app_name)
url = f"{app_info.url}/api/sync"

payload = {
    "locations": ["Austin, TX"],
    "limit": 10
}

response = requests.post(
    url,
    json=payload,
    headers={
        "Authorization": f"Bearer {app_token}",
        "Content-Type": "application/json",
    },
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))